<a href="https://colab.research.google.com/github/Mihret4204/math4ai/blob/main/Mihret_Shewarega_Math_for_AI_Lab_4_2019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Name:  **Mihret Tena**


#ID:  **UGR/4204/16**


# 📄 **Lab 4: Text Data Representation Using Vectors and Matrices**

**Course:** Mathematics for Artificial Intelligence

## Introduction

Text is one of the most common forms of data used in artificial intelligence. Examples include books, articles, emails, customer reviews, and conversations.

Unlike numerical data, raw text consists of characters, words, and sentences. To process text using mathematical algorithms, we must first transform it into numerical representations.

In this laboratory, we will work with a real text document and explore how text can be represented using **vectors and matrices**.

We will begin by downloading and examining the document, then progressively transform its contents into numerical representations suitable for further analysis and machine learning.

### Learning Objectives

By the end of this laboratory, you should be able to:

1. Download and load a real text document using Python.
2. Examine the structure of text data, including characters, words, sentences, and paragraphs.
3. Apply basic text preprocessing and tokenization.
4. Construct a vocabulary and assign numerical identifiers to words.
5. Represent words using integer encoding and one-hot vectors.
6. Represent documents using binary and count-based Bag-of-Words vectors.
7. Construct document-term matrices from multiple text documents.
8. Prepare fixed-length numerical sequences using padding and truncation.


In [ ]:

#@title Lab 4 Utilities (Run Once) { display-mode: "form" }

import re
import string
import numpy as np
import matplotlib.pyplot as plt

from urllib.request import urlopen
from collections import Counter

def show_documents(documents, max_chars=150):
    """Display a numbered collection of text documents."""
    for i, doc in enumerate(documents, start=1):
        preview = doc[:max_chars].replace("\n", " ")
        print(f"Document {i}: {preview}")


# **Part A: Loading and Exploring Text Data**

Before representing text numerically, we need to understand how text is stored and organized.

A text document can contain several structural elements:

- **Characters:** Individual letters, digits, punctuation marks, and spaces.
- **Words:** Sequences of characters that form meaningful units.
- **Sentences:** Groups of words that express statements or ideas.
- **Paragraphs:** Groups of related sentences.
- **Documents:** Complete pieces of text, such as books, articles, or reports.

In Python, text is commonly stored as a string (`str`). A string is a sequence of characters that can be accessed using indexing and slicing.

We will first download a real document and examine its basic properties.


### **Laboratory Dataset**

We will use **Alice's Adventures in Wonderland** by Lewis Carroll, available through [Project Gutenberg](https://www.gutenberg.org/ebooks/11).

The document contains natural-language text with sentences, paragraphs, punctuation, repeated words, and varying sentence lengths.

We will use this document throughout the laboratory to demonstrate how real text is transformed into numerical data.

In [ ]:

# Downloading a Real Text Document

# Source: GitHub (dakrone/corpus)
# Book: Alice's Adventures in Wonderland

url = (
    "https://raw.githubusercontent.com/dakrone/corpus/"
    "master/data/alice-in-wonderland.txt"
)

with urlopen(url, timeout=30) as response:
    raw_text = response.read().decode("utf-8-sig")


## **Exploring Characters, Lines, and Paragraphs**

A text document is stored in Python as a string (`str`), which is a sequence of characters. If a string contains $N$ characters, its positions are indexed from $0$ to $N-1$.

We can use indexing and slicing to access characters or extract portions of text, similar to accessing elements of a numerical vector. However, characters are textual symbols, not numerical vector components.

Text documents also contain structural elements:

- **Lines:** Text separated by newline characters (`\n`).
- **Paragraphs:** Groups of text commonly separated by blank lines.
- **Documents:** Complete pieces of text, such as books or articles.

We can extract paragraph-like blocks using `re.split(r"\n\s*\n", text)`, which splits text at blank lines.

In this demonstration, we will inspect characters, lines, paragraphs, and basic statistics from the downloaded document.

**Note:** Line breaks and blank lines depend on document formatting and do not always correspond to sentence or paragraph boundaries.


In [ ]:
# Exploring Text Structure
def extract_paragraphs(text):
    """Split text into nonempty blocks separated by blank lines."""

    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

In [ ]:
def show_paragraphs(paragraphs, start=0, end=1):
    """Display paragraphs with their internal line breaks preserved."""

    for paragraph in paragraphs[start:end]:
        print(paragraph)
        print()  # Blank line between paragraphs

In [ ]:
# 0. Preview the document

print("Data type:", type(raw_text))
print("\nBeginning of the downloaded document:\n\n\n", raw_text[:800])

Data type: <class 'str'>

Beginning of the downloaded document:


 Project Gutenberg's Alice's Adventures in Wonderland, by Lewis Carroll

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org


Title: Alice's Adventures in Wonderland

Author: Lewis Carroll

Posting Date: June 25, 2008 [EBook #11]
Release Date: March, 1994

Language: English


*** START OF THIS PROJECT GUTENBERG EBOOK ALICE'S ADVENTURES IN WONDERLAND ***










ALICE'S ADVENTURES IN WONDERLAND

Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0




CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, 


In [ ]:
# 1. Character indexing and slicing
print("First character:", repr(raw_text[0]))
print("First 50 characters:", repr(raw_text[:50]))
print("Last 100 characters:", repr(raw_text[-100:]))

First character: 'P'
First 50 characters: "Project Gutenberg's Alice's Adventures in Wonderla"
Last 100 characters: 'lp produce our new eBooks, and how to\r\nsubscribe to our email newsletter to hear about new eBooks.\r\n'


In [ ]:
# 2. Basic text statistics
lines = raw_text.splitlines()
raw_paragraphs = extract_paragraphs(raw_text)

In [ ]:
print("\nTotal characters:", len(raw_text))
print("Number of spaces:", raw_text.count(" "))
print("Number of newline characters:", raw_text.count("\n"))
print("Number of lines:", len(lines))
print("Number of paragraph-like blocks:", len(raw_paragraphs))



Total characters: 167497
Number of spaces: 27982
Number of newline characters: 3735
Number of lines: 3735
Number of paragraph-like blocks: 881


In [ ]:
# 3. Inspect the extracted blocks
print("\nSample paragraph-like blocks:\n")
show_paragraphs(raw_paragraphs, start=25, end=28)


Sample paragraph-like blocks:

There seemed to be no use in waiting by the little door, so she went
back to the table, half hoping she might find another key on it, or at
any rate a book of rules for shutting people up like telescopes: this
time she found a little bottle on it, ('which certainly was not here
before,' said Alice,) and round the neck of the bottle was a paper
label, with the words 'DRINK ME' beautifully printed on it in large
letters.

It was all very well to say 'Drink me,' but the wise little Alice was
not going to do THAT in a hurry. 'No, I'll look first,' she said, 'and
see whether it's marked "poison" or not'; for she had read several nice
little histories about children who had got burnt, and eaten up by wild
beasts and other unpleasant things, all because they WOULD not remember
the simple rules their friends had taught them: such as, that a red-hot
poker will burn you if you hold it too long; and that if you cut your
finger VERY deeply with a knife, it usually b


## **Cleaning and Preparing the Text Dataset**

Real-world text documents often contain metadata, formatting elements, and other information that may not be relevant to a particular machine learning task.

For example, our Project Gutenberg document contains introductory metadata and licensing information outside the book's content.

We will prepare the dataset in three steps:

1. **Remove metadata:** Locate the Project Gutenberg start and end markers and extract the book's content.
2. **Extract and analyze paragraphs:** Divide the cleaned book into paragraph-like blocks and examine their lengths.
3. **Select narrative paragraphs:** Choose a small collection of paragraphs for our later numerical representations.

For a collection of $D$ paragraphs, their lengths can be represented as a numerical vector:

$$
\boldsymbol{\ell}=
[\ell_1,\ell_2,\ldots,\ell_D]
$$

where $\ell_i$ is the number of whitespace-separated tokens in paragraph $i$.

We will preserve the complete extracted book and use a smaller subset of narrative paragraphs for the remaining demonstrations and exercises.

**Note:** The extracted blocks may include titles, chapter headings, or illustration placeholders. Selecting relevant content is therefore an important part of preparing real-world text data.


In [ ]:

# Cleaning and Preparing the Text Dataset

def remove_gutenberg_metadata(text):
    """Extract the content between Project Gutenberg markers."""

    pattern = (
        r"^\*\*\* (START|END) OF (?:THE|THIS) "
        r"PROJECT GUTENBERG EBOOK[^\n]*\*\*\*\s*$"
    )

    markers = list(re.finditer(
        pattern, text, flags=re.MULTILINE | re.IGNORECASE
    ))

    starts = [m for m in markers if m.group(1).upper() == "START"]
    ends = [m for m in markers if m.group(1).upper() == "END"]

    if not starts or not ends:
        raise ValueError("Project Gutenberg markers not found.")

    start = starts[0].end()
    end = next(
        (m.start() for m in ends if m.start() > start),
        None
    )

    if end is None:
        raise ValueError("No valid end marker found.")

    return text[start:end].strip()


In [ ]:
# 1. Remove external metadata
book_text = remove_gutenberg_metadata(raw_text)

# 2. Extract paragraphs from the book
paragraphs = extract_paragraphs(book_text)

print("Original characters:", len(raw_text))
print("Extracted book characters:", len(book_text))
print("Number of extracted blocks:", len(paragraphs))

print("\nFirst 5 extracted blocks:\n")
show_paragraphs(paragraphs, start=0, end=5)

Original characters: 167497
Extracted book characters: 147875
Number of extracted blocks: 821

First 5 extracted blocks:

ALICE'S ADVENTURES IN WONDERLAND

Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversation?'



In [ ]:
# 3. Examine paragraph lengths
paragraph_lengths = np.array([
    len(p.split()) for p in paragraphs
])

print("\nParagraph length statistics:")
print("Minimum:", paragraph_lengths.min())
print("Maximum:", paragraph_lengths.max())
print("Mean:", round(paragraph_lengths.mean(), 2))
print("Median:", np.median(paragraph_lengths))


Paragraph length statistics:
Minimum: 1
Maximum: 191
Mean: 32.23
Median: 21.0


In [ ]:
# 4. Select a small collection of narrative paragraphs
start_index = 18

selected_paragraphs = paragraphs[start_index:start_index + 12]

print("\nSelected narrative paragraphs:", len(selected_paragraphs))
print("\nFirst 3 selected paragraphs:\n")
show_paragraphs(selected_paragraphs, start=0, end=3)


Selected narrative paragraphs: 12

First 3 selected paragraphs:

There seemed to be no use in waiting by the little door, so she went
back to the table, half hoping she might find another key on it, or at
any rate a book of rules for shutting people up like telescopes: this
time she found a little bottle on it, ('which certainly was not here
before,' said Alice,) and round the neck of the bottle was a paper
label, with the words 'DRINK ME' beautifully printed on it in large
letters.

It was all very well to say 'Drink me,' but the wise little Alice was
not going to do THAT in a hurry. 'No, I'll look first,' she said, 'and
see whether it's marked "poison" or not'; for she had read several nice
little histories about children who had got burnt, and eaten up by wild
beasts and other unpleasant things, all because they WOULD not remember
the simple rules their friends had taught them: such as, that a red-hot
poker will burn you if you hold it too long; and that if you cut your
finger VERY


# **Part B: Text Preprocessing and Vocabulary Construction**

Raw text contains letters, punctuation, spaces, line breaks, and other characters. Before representing text as numerical vectors and matrices, we need to organize it into consistent units.

**Text preprocessing** refers to operations that prepare raw text for further analysis and machine learning.

In this part, we will:

1. Normalize text by converting letters to lowercase and handling punctuation.
2. Split paragraphs into individual words using tokenization.
3. Construct a vocabulary containing the unique words in our dataset.
4. Assign a numerical identifier to each vocabulary word.
5. Examine the vocabulary size and word frequencies.

We will use the narrative paragraphs selected in Part A.

**Important:** Text preprocessing is task-dependent. Lowercasing and removing punctuation are useful for our introductory demonstrations, but they are not always appropriate for every NLP application.



## **Text Normalization**

**Text normalization** transforms text into a more consistent format.

For example, the words `Alice`, `ALICE`, and `alice` have different character representations. Converting them to lowercase allows us to treat them as the same word.

Similarly, punctuation can create differences between tokens such as `rabbit`, `rabbit,`, and `rabbit.`.

For this demonstration, we will:

- Convert text to lowercase.
- Replace punctuation with spaces.
- Replace repeated whitespace with a single space.

We will preserve the original paragraphs and store the normalized versions separately.

Replacing punctuation with spaces, rather than deleting it, helps avoid accidentally joining words together.

**Note:** Removing punctuation also removes information, such as sentence boundaries and distinctions between some expressions. We are using a simplified normalization procedure to introduce numerical text representation.


In [ ]:

# Text Normalization

def normalize_text(text):
    """Lowercase text, replace punctuation, and normalize whitespace."""

    # Convert uppercase letters to lowercase.
    text = text.lower()

    # Replace punctuation and other non-word symbols with spaces.
    # \w matches Unicode word characters, including letters,
    # digits, and underscores; \s matches whitespace.
    text = re.sub(r"[^\w\s]", " ", text)

    # Replace underscores with spaces.
    text = text.replace("_", " ")

    # Replace consecutive whitespace characters with one space.
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
# Normalize the selected paragraphs.
normalized_paragraphs = [normalize_text(p) for p in selected_paragraphs]

In [ ]:
print("Original paragraph:\n")
print(selected_paragraphs[0])

Original paragraph:

There seemed to be no use in waiting by the little door, so she went
back to the table, half hoping she might find another key on it, or at
any rate a book of rules for shutting people up like telescopes: this
time she found a little bottle on it, ('which certainly was not here
before,' said Alice,) and round the neck of the bottle was a paper
label, with the words 'DRINK ME' beautifully printed on it in large
letters.


In [ ]:
print("\nNormalized paragraph:\n")
print(normalized_paragraphs[0])


Normalized paragraph:

there seemed to be no use in waiting by the little door so she went back to the table half hoping she might find another key on it or at any rate a book of rules for shutting people up like telescopes this time she found a little bottle on it which certainly was not here before said alice and round the neck of the bottle was a paper label with the words drink me beautifully printed on it in large letters



## **Word Tokenization**

**Tokenization** is the process of dividing text into smaller units called *tokens*.

Tokens may represent words, punctuation marks, characters, or other text units, depending on the tokenization method.

For this laboratory, we will use a simple word-level approach.

Consider the sentence:

`Alice followed the white rabbit.`

After normalization:

`alice followed the white rabbit`

Tokenization produces:

```python
["alice", "followed", "the", "white", "rabbit"]
```

If paragraph $i$ contains $n_i$ tokens, we can represent its token sequence as:

$$
T_i=(t_{i,1},t_{i,2},\ldots,t_{i,n_i})
$$

where $t_{i,j}$ is the token at position $j$ in paragraph $i$.

At this stage, the tokens are still strings. We will convert them into numerical identifiers in a later demonstration.


In [ ]:

# Word Tokenization

# Split each normalized paragraph into words.
tokenized_paragraphs = [
    p.split() for p in normalized_paragraphs
]


In [ ]:
print("First normalized paragraph:\n")
print(normalized_paragraphs[0])

First normalized paragraph:

there seemed to be no use in waiting by the little door so she went back to the table half hoping she might find another key on it or at any rate a book of rules for shutting people up like telescopes this time she found a little bottle on it which certainly was not here before said alice and round the neck of the bottle was a paper label with the words drink me beautifully printed on it in large letters


In [ ]:
print("\nFirst 20 tokens:\n")
print(tokenized_paragraphs[0][:20])


First 20 tokens:

['there', 'seemed', 'to', 'be', 'no', 'use', 'in', 'waiting', 'by', 'the', 'little', 'door', 'so', 'she', 'went', 'back', 'to', 'the', 'table', 'half']


In [ ]:
print("\nNumber of tokens:", len(tokenized_paragraphs[0]))


Number of tokens: 82


In [ ]:
# Examine token counts across all selected paragraphs.
token_counts = np.array([
    len(tokens) for tokens in tokenized_paragraphs
])

In [ ]:
print("\nTotal tokens:", token_counts.sum())
print("Minimum tokens per paragraph:", token_counts.min())
print("Maximum tokens per paragraph:", token_counts.max())
print("Average tokens per paragraph:", round(token_counts.mean(), 2))


Total tokens: 784
Minimum tokens per paragraph: 0
Maximum tokens per paragraph: 134
Average tokens per paragraph: 65.33



## **Vocabulary Construction**

A **vocabulary** is the collection of unique tokens found in a text dataset.

For example, consider two sentences:

- `alice sees a rabbit`
- `the rabbit sees alice`

Their combined vocabulary is:

$$
V=\{\text{a},\text{alice},\text{rabbit},\text{sees},\text{the}\}
$$

The vocabulary contains five unique words, even though some words appear more than once.

The **vocabulary size** is the number of unique tokens:

$$
|V|=5
$$

For consistent numerical encoding, we will arrange the vocabulary alphabetically.

We can then construct a mapping that assigns each vocabulary word a unique integer identifier.


In [ ]:

# Vocabulary Construction

# Combine tokens from all selected paragraphs.
all_tokens = [token for paragraph in tokenized_paragraphs for token in paragraph]


In [ ]:
# Extract unique tokens and sort them alphabetically.
vocabulary = sorted(set(all_tokens))

print("Total tokens:", len(all_tokens))
print("Unique tokens:", len(vocabulary))

print("\nFirst 30 vocabulary words:")
print(vocabulary[:30])

print("\nLast 10 vocabulary words:")
print(vocabulary[-10:])

Total tokens: 784
Unique tokens: 316

First 30 vocabulary words:
['a', 'about', 'advice', 'advise', 'after', 'against', 'alas', 'alice', 'all', 'almost', 'altogether', 'and', 'another', 'anxiously', 'any', 'apple', 'as', 'at', 'ate', 'back', 'be', 'beasts', 'beautifully', 'because', 'before', 'best', 'bit', 'bleeds', 'blown', 'book']

Last 10 vocabulary words:
['why', 'wild', 'will', 'wise', 'with', 'wonder', 'words', 'would', 'you', 'your']



## **Assigning Numerical Identifiers to Words**

AI algorithms generally operate on numerical data rather than raw strings.

One simple way to represent words numerically is to assign each vocabulary word a unique integer identifier.

For a vocabulary containing $V$ words, we can define a mapping:

$$
f:V\rightarrow\{1,2,\ldots,V\}
$$

where ${V}$ is the set of vocabulary words.

For example:

| Word | Integer ID |
|---|---:|
| a | 1 |
| alice | 2 |
| rabbit | 3 |
| sees | 4 |
| the | 5 |

We reserve the integer `0` for padding, which will be introduced later when preparing fixed-length sequences.

The integer identifiers are **categorical labels**, not numerical measurements. For example, word ID 4 does not mean that the corresponding word has twice the meaning or importance of word ID 2.

We can use the mapping to convert each paragraph into a sequence of integers while preserving its token order.


In [ ]:

# Word-to-Index Mapping

# Assign IDs starting from 1; reserve 0 for padding.
word_to_id = {word: i + 1 for i, word in enumerate(vocabulary)}


In [ ]:
# Reverse mapping: integer ID to word.
id_to_word = {i: word for word, i in word_to_id.items()}

In [ ]:
# Encode each paragraph as a sequence of integer IDs.
encoded_paragraphs = [
    [word_to_id[token] for token in tokens]
    for tokens in tokenized_paragraphs
]

In [ ]:
print("Vocabulary size:", len(vocabulary))

print("\nFirst 10 word-to-ID mappings:")
for word in vocabulary[:10]:
    print(f"{word}: {word_to_id[word]}")

print("\nFirst 15 tokens:")
print(tokenized_paragraphs[0][:15])

print("\nCorresponding integer IDs:")
print(encoded_paragraphs[0][:15])

Vocabulary size: 316

First 10 word-to-ID mappings:
a: 1
about: 2
advice: 3
advise: 4
after: 5
against: 6
alas: 7
alice: 8
all: 9
almost: 10

First 15 tokens:
['there', 'seemed', 'to', 'be', 'no', 'use', 'in', 'waiting', 'by', 'the', 'little', 'door', 'so', 'she', 'went']

Corresponding integer IDs:
[268, 230, 278, 21, 181, 290, 140, 295, 39, 264, 159, 67, 245, 236, 299]


In [ ]:
# Verify that we can recover the original tokens.
decoded_tokens = [
    id_to_word[i] for i in encoded_paragraphs[0][:15]
]

In [ ]:
print("\nDecoded tokens:")
print(decoded_tokens)


Decoded tokens:
['there', 'seemed', 'to', 'be', 'no', 'use', 'in', 'waiting', 'by', 'the', 'little', 'door', 'so', 'she', 'went']



## **Exploring Word Frequencies**

Some words occur more frequently than others in a text dataset.

The **frequency** of a word is the number of times it appears in the collection.

If a word $w$ appears in the token collection $T$, its frequency is:

$$
f(w)=\sum_{t\in T}\mathbf{1}(t=w)
$$

where $\mathbf{1}(t=w)$ equals 1 when token $t$ is equal to $w$, and 0 otherwise.

We can calculate these frequencies to understand the distribution of words in our selected paragraphs.

This will also prepare us for count-based Bag-of-Words representations in Part C.


In [ ]:

# Word Frequencies

word_frequencies = Counter(all_tokens)

# Identify the 15 most frequent words.
most_common = word_frequencies.most_common(15)

print("15 most frequent words:\n")
for word, count in most_common:
    print(f"{word:<15} {count}")

15 most frequent words:

she             33
the             30
to              27
it              27
and             24
a               20
was             15
of              13
for             11
that            11
in              10
little          10
alice           10
i               10
on              8



## ✍️ Exercise 1: Text Preprocessing and Vocabulary Construction

In this exercise, you will apply the text preprocessing and vocabulary construction techniques introduced in Part B to a new collection of paragraphs from *Alice's Adventures in Wonderland*.

### Tasks

1. **Select text:** Select five consecutive paragraphs from `paragraphs` that were not included in `selected_paragraphs`. Display the selected paragraphs using `show_paragraphs()`.
2. **Normalize text:** Convert the paragraphs to lowercase, replace punctuation with spaces, and remove unnecessary whitespace.
3. **Tokenize:** Split each normalized paragraph into a list of words.
4. **Construct a vocabulary:** Identify and alphabetically sort the unique words in the five paragraphs.
5. **Assign word IDs:** Create a dictionary mapping each vocabulary word to a unique integer ID, starting from 1. Reserve 0 for padding.
6. **Encode and decode:** Convert the first paragraph into a sequence of integer IDs. Decode the first 15 IDs to verify the encoding.
7. **Analyze word frequencies:** Identify and display the 10 most frequent words in the selected paragraphs.

### Hints

- Use list slicing to select paragraphs, for example, `paragraphs[start:end]`.
- Use `.lower()` to convert text to lowercase.
- Use `re.sub()` to replace punctuation and repeated whitespace.
- Use `.split()` to tokenize normalized text.
- Use `set()` to obtain unique words and `sorted()` to arrange them alphabetically.
- Use `enumerate()` to assign integer IDs.
- Use `Counter()` to count word occurrences.
- Remember that the same word must always receive the same integer ID.


In [ ]:

# ============================================================
# Exercise 1: Text Preprocessing and Vocabulary Construction
# ============================================================

# ------------------------------------------------------------
# Task 1: Select five new paragraphs
# ------------------------------------------------------------

# TODO: Choose five consecutive paragraphs not used previously.
start = 10  # Replace with a valid starting index
end = 15   # Replace with the ending index

exercise_paragraphs = paragraphs[start:end]

print("Selected paragraphs:")
# print(exercise_paragraphs)
show_paragraphs(exercise_paragraphs, start=0, end=5)

Selected paragraphs:
'Well!' thought Alice to herself, 'after such a fall as this, I shall
think nothing of tumbling down stairs! How brave they'll all think me at
home! Why, I wouldn't say anything about it, even if I fell off the top
of the house!' (Which was very likely true.)

Down, down, down. Would the fall NEVER come to an end! 'I wonder how
many miles I've fallen by this time?' she said aloud. 'I must be getting
somewhere near the centre of the earth. Let me see: that would be four
thousand miles down, I think--' (for, you see, Alice had learnt several
things of this sort in her lessons in the schoolroom, and though this
was not a VERY good opportunity for showing off her knowledge, as there
was no one to listen to her, still it was good practice to say it over)
'--yes, that's about the right distance--but then I wonder what Latitude
or Longitude I've got to?' (Alice had no idea what Latitude was, or
Longitude either, but thought they were nice grand words to say.)

Presently s

In [ ]:
# ------------------------------------------------------------
# Task 2: Normalize the selected paragraphs
# ------------------------------------------------------------

exercise_normalized = []



for paragraph in exercise_paragraphs:

    # TODO: Convert the paragraph to lowercase.
    text = paragraph.lower()

    # TODO: Replace punctuation with spaces using re.sub().
    text = re.sub(r"[^\w\s]"," ",text)

    # TODO: Replace repeated whitespace with a single space.
    text = re.sub(r"\s+"," ",text)

    # Remove leading and trailing whitespace.
    text = text.strip()

    exercise_normalized.append(text)

print("\nFirst normalized paragraph:")
exercise_normalized[0]


First normalized paragraph:


'well thought alice to herself after such a fall as this i shall think nothing of tumbling down stairs how brave they ll all think me at home why i wouldn t say anything about it even if i fell off the top of the house which was very likely true'

In [ ]:
# ------------------------------------------------------------
# Task 3: Tokenize the paragraphs
# ------------------------------------------------------------

# TODO: Split each normalized paragraph into words.
exercise_tokens = [p.split() for p in exercise_normalized ]
print("\nFirst 20 tokens: ",exercise_tokens[0][:20] )



First 20 tokens:  ['well', 'thought', 'alice', 'to', 'herself', 'after', 'such', 'a', 'fall', 'as', 'this', 'i', 'shall', 'think', 'nothing', 'of', 'tumbling', 'down', 'stairs', 'how']


In [ ]:
# ------------------------------------------------------------
# Task 4: Construct the vocabulary
# ------------------------------------------------------------

# TODO: Combine all tokens into a single list.
exercise_all_tokens = [token  for paras in exercise_tokens for token in paras ]

# TODO: Extract unique words and sort alphabetically.
exercise_vocabulary = sorted(set(exercise_all_tokens))
print(exercise_all_tokens)
print("\nTotal tokens:", len(exercise_all_tokens))
print("Vocabulary size:", len(exercise_vocabulary))

['well', 'thought', 'alice', 'to', 'herself', 'after', 'such', 'a', 'fall', 'as', 'this', 'i', 'shall', 'think', 'nothing', 'of', 'tumbling', 'down', 'stairs', 'how', 'brave', 'they', 'll', 'all', 'think', 'me', 'at', 'home', 'why', 'i', 'wouldn', 't', 'say', 'anything', 'about', 'it', 'even', 'if', 'i', 'fell', 'off', 'the', 'top', 'of', 'the', 'house', 'which', 'was', 'very', 'likely', 'true', 'down', 'down', 'down', 'would', 'the', 'fall', 'never', 'come', 'to', 'an', 'end', 'i', 'wonder', 'how', 'many', 'miles', 'i', 've', 'fallen', 'by', 'this', 'time', 'she', 'said', 'aloud', 'i', 'must', 'be', 'getting', 'somewhere', 'near', 'the', 'centre', 'of', 'the', 'earth', 'let', 'me', 'see', 'that', 'would', 'be', 'four', 'thousand', 'miles', 'down', 'i', 'think', 'for', 'you', 'see', 'alice', 'had', 'learnt', 'several', 'things', 'of', 'this', 'sort', 'in', 'her', 'lessons', 'in', 'the', 'schoolroom', 'and', 'though', 'this', 'was', 'not', 'a', 'very', 'good', 'opportunity', 'for', 'sho

In [ ]:
# ------------------------------------------------------------
# Task 5: Assign integer IDs to vocabulary words
# ------------------------------------------------------------

# TODO: Create a word-to-ID dictionary starting from 1.
exercise_word_to_id = {word: i + 1 for i, word in enumerate(exercise_vocabulary)}

# TODO: Create the reverse ID-to-word dictionary.
exercise_id_to_word =  {i: word for word, i in exercise_word_to_id.items()}

print("\nFirst 10 word-to-ID mappings:")
for j in range(5):
  print(exercise_id_to_word[j+1])


First 10 word-to-ID mappings:
a
about
afraid
after
again


In [ ]:
# ------------------------------------------------------------
# Task 6: Encode and decode the first paragraph
# ------------------------------------------------------------

# TODO: Convert the first tokenized paragraph into integer IDs.
exercise_encoded = [[exercise_word_to_id[token] for token in tokens]
    for tokens in exercise_tokens]

# TODO: Decode the first 15 IDs back into words.
exercise_decoded = [exercise_id_to_word[i] for i in exercise_encoded[0][:15]]

print("\nFirst 15 original tokens:",exercise_tokens)


print("\nFirst 15 encoded IDs:", exercise_encoded)


print("\nDecoded tokens:",exercise_decoded)


# Verify that encoding and decoding agree.
assert exercise_decoded == exercise_tokens[0][:15]


First 15 original tokens: [['well', 'thought', 'alice', 'to', 'herself', 'after', 'such', 'a', 'fall', 'as', 'this', 'i', 'shall', 'think', 'nothing', 'of', 'tumbling', 'down', 'stairs', 'how', 'brave', 'they', 'll', 'all', 'think', 'me', 'at', 'home', 'why', 'i', 'wouldn', 't', 'say', 'anything', 'about', 'it', 'even', 'if', 'i', 'fell', 'off', 'the', 'top', 'of', 'the', 'house', 'which', 'was', 'very', 'likely', 'true'], ['down', 'down', 'down', 'would', 'the', 'fall', 'never', 'come', 'to', 'an', 'end', 'i', 'wonder', 'how', 'many', 'miles', 'i', 've', 'fallen', 'by', 'this', 'time', 'she', 'said', 'aloud', 'i', 'must', 'be', 'getting', 'somewhere', 'near', 'the', 'centre', 'of', 'the', 'earth', 'let', 'me', 'see', 'that', 'would', 'be', 'four', 'thousand', 'miles', 'down', 'i', 'think', 'for', 'you', 'see', 'alice', 'had', 'learnt', 'several', 'things', 'of', 'this', 'sort', 'in', 'her', 'lessons', 'in', 'the', 'schoolroom', 'and', 'though', 'this', 'was', 'not', 'a', 'very', 'goo

In [ ]:
# ------------------------------------------------------------
# Task 7: Analyze word frequencies
# ------------------------------------------------------------

# TODO: Count the frequency of each word.
exercise_frequencies = Counter(exercise_all_tokens)

# TODO: Identify the 10 most frequent words.
top_10 = exercise_frequencies.most_common(10)

print("\n10 most frequent words:",top_10)



10 most frequent words: [('the', 23), ('to', 21), ('was', 21), ('i', 19), ('she', 17), ('it', 16), ('and', 15), ('a', 13), ('down', 11), ('you', 10)]



# **Part C: Representing Text as Vectors and Matrices**

In Part B, we converted text into tokens, constructed a vocabulary, and assigned integer identifiers to words.

However, integer IDs are categorical labels, not numerical measurements. For example, a word with ID 20 is not necessarily more important than a word with ID 5.

To prepare text for machine learning, we can construct numerical **vectors and matrices** that represent words, documents, or sequences of tokens.

In this part, we will explore five representations:

| Representation | What it represents | Example applications |
|---|---|---|
| One-hot encoding | An individual word as a binary vector | Categorical word encoding, embedding lookup concepts |
| Binary Bag-of-Words | Presence or absence of vocabulary words | Spam detection, document classification |
| Count Bag-of-Words | Frequency of vocabulary words | Spam detection, sentiment analysis, topic modeling |
| Document-term matrix | A collection of documents represented by word features | Classical machine learning classifiers |
| Padded token-ID matrix | Ordered sequences of token identifiers | Neural networks, embedding-based sequence models |

**Learning objective:** Understand how real text is transformed into numerical data and how the resulting representations can serve as inputs to machine learning systems.

We will use the vocabulary and selected paragraphs from *Alice's Adventures in Wonderland* prepared in Part B.

-------
-------

## **One-Hot and Bag-of-Words Representations**

Suppose our vocabulary contains $V$ unique words:

$$
V=\{w_1,w_2,\ldots,w_V\}
$$

We can construct three types of vectors.

**1. One-hot encoding**

An individual vocabulary word is represented by a vector containing exactly one component equal to 1:

$$
\mathbf{e}_j\in\{0,1\}^{V}
$$

where the value 1 indicates the position of word $w_j$.

**2. Binary Bag-of-Words (BoW)**

A document is represented by a vector indicating whether each vocabulary word appears:

$$
b_j=
\begin{cases}
1, & \text{if word }w_j\text{ appears}\\
0, & \text{otherwise}
\end{cases}
$$

**3. Count Bag-of-Words**

A document is represented by a vector containing the frequency of each vocabulary word:

$$
c_j=\text{number of occurrences of word }w_j
$$

All three representations have $V$ components, but they capture different information.

We will construct these vectors using a paragraph from our book.

**Important:** In Part B, word IDs start from 1 because 0 is reserved for padding. NumPy vector positions start from 0, so the position of a word is `word_to_id[word] - 1`.

**Applications:** Binary and count BoW vectors can be used as features for spam detection, sentiment classification, and document categorization. One-hot vectors are useful for representing categorical word identities but become inefficient for large vocabularies.

**Limitation:** Bag-of-Words does not preserve the original order of words.


In [ ]:

# One-Hot, Binary BoW, and Count BoW

# Use the first selected paragraph.
tokens = tokenized_paragraphs[0]
V = len(vocabulary)

print("Vocabulary size:", V)

Vocabulary size: 316


In [ ]:
# 1. One-hot encoding for a selected word
word = "alice"

one_hot = np.zeros(V, dtype=int)
one_hot[word_to_id[word] - 1] = 1

print("\nOne-hot representation:")
print("Word:", word)
print("Vector shape:", one_hot.shape)
print("Nonzero position:", np.nonzero(one_hot)[0])
print("One-hot vector:\n", one_hot)


One-hot representation:
Word: alice
Vector shape: (316,)
Nonzero position: [7]
One-hot vector:
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
# 2. Binary and count Bag-of-Words
binary_bow = np.zeros(V, dtype=int)
count_bow = np.zeros(V, dtype=int)

for token in tokens:
    j = word_to_id[token] - 1
    binary_bow[j] = 1
    count_bow[j] += 1


print("\nBag-of-Words representations:")
print("Binary vector shape:", binary_bow.shape)
print("Count vector shape:", count_bow.shape)

print("\nFirst Tokenized paragraph sample:\n", tokenized_paragraphs[0][:30])
print("\nInteger encoded words sample:\n", word_to_id)
print("\n\nBag-of-Words sample:\n", binary_bow[:30])
print("\n\nCount Bag-of-Words sample:\n", count_bow[:30])



Bag-of-Words representations:
Binary vector shape: (316,)
Count vector shape: (316,)

First Tokenized paragraph sample:
 ['there', 'seemed', 'to', 'be', 'no', 'use', 'in', 'waiting', 'by', 'the', 'little', 'door', 'so', 'she', 'went', 'back', 'to', 'the', 'table', 'half', 'hoping', 'she', 'might', 'find', 'another', 'key', 'on', 'it', 'or', 'at']

Integer encoded words sample:
 {'a': 1, 'about': 2, 'advice': 3, 'advise': 4, 'after': 5, 'against': 6, 'alas': 7, 'alice': 8, 'all': 9, 'almost': 10, 'altogether': 11, 'and': 12, 'another': 13, 'anxiously': 14, 'any': 15, 'apple': 16, 'as': 17, 'at': 18, 'ate': 19, 'back': 20, 'be': 21, 'beasts': 22, 'beautifully': 23, 'because': 24, 'before': 25, 'best': 26, 'bit': 27, 'bleeds': 28, 'blown': 29, 'book': 30, 'bottle': 31, 'box': 32, 'brightened': 33, 'bring': 34, 'burn': 35, 'burnt': 36, 'but': 37, 'buttered': 38, 'by': 39, 'cake': 40, 'can': 41, 'candle': 42, 'care': 43, 'certain': 44, 'certainly': 45, 'cheated': 46, 'cherry': 47, 'child':

In [ ]:
print("\nWord frequencies in the paragraph:")
print(f"{'Word':<18} {'Binary':<10} {'Count':<10}")

for j in np.nonzero(binary_bow)[0]:
    print(
        f"{vocabulary[j]:<18} "
        f"{binary_bow[j]:<10} "
        f"{count_bow[j]:<10}"
    )

print("\nTotal tokens:", len(tokens))
print("Sum of count vector:", count_bow.sum())


Word frequencies in the paragraph:
Word               Binary     Count     
a                  1          3         
alice              1          1         
and                1          1         
another            1          1         
any                1          1         
at                 1          1         
back               1          1         
be                 1          1         
beautifully        1          1         
before             1          1         
book               1          1         
bottle             1          2         
by                 1          1         
certainly          1          1         
door               1          1         
drink              1          1         
find               1          1         
for                1          1         
found              1          1         
half               1          1         
here               1          1         
hoping             1          1         
in                 1 


## **Constructing a Document-Term Matrix**

In many machine learning applications, we work with collections of documents rather than individual paragraphs.

A **document-term matrix (DTM)** organizes the Bag-of-Words vectors of multiple documents into a single matrix.

Suppose we have $D$ documents and a vocabulary containing $V$ words.

The count-based document-term matrix is:

$$
X=
\begin{bmatrix}
\mathbf{x}_1^T\\
\mathbf{x}_2^T\\
\vdots\\
\mathbf{x}_D^T
\end{bmatrix}
\in\mathbb{R}^{D\times V}
$$

where each row is a document's count vector.

The matrix entry $X_{ij}$ records how many times vocabulary word $j$ appears in document $i$.

A **binary document-term matrix** has the same dimensions, but its entries indicate only word presence or absence.

For example:

$$
X=
\begin{bmatrix}
2&0&1\\
0&1&2\\
1&1&0
\end{bmatrix}
$$

represents three documents using three vocabulary words.

We will construct both matrices using paragraphs from our book.

**Applications:** Document-term matrices are commonly used to prepare text features for classical machine learning algorithms such as Naive Bayes, logistic regression, and support vector machines.

**Important:** In practical applications, document-term matrices are often stored as sparse matrices because most documents contain only a small fraction of the complete vocabulary.


In [ ]:

# Document-Term Matrices

# Use the first five selected paragraphs.
documents = tokenized_paragraphs[:5]

D = len(documents)
V = len(vocabulary)

print("Number of documents:", D)
print("Vocabulary size:", V)

Number of documents: 5
Vocabulary size: 316


In [ ]:
# Construct the count document-term matrix.
count_matrix = np.zeros((D, V), dtype=int)

for i, tokens in enumerate(documents):
    for token in tokens:
        j = word_to_id[token] - 1
        count_matrix[i, j] += 1

# Construct the binary document-term matrix.
binary_matrix = (count_matrix > 0).astype(int)


print("Count matrix shape:", count_matrix.shape)
print("Binary matrix shape:", binary_matrix.shape)

Count matrix shape: (5, 316)
Binary matrix shape: (5, 316)


In [ ]:
count_matrix

array([[3, 0, 0, ..., 0, 0, 0],
       [4, 1, 0, ..., 1, 5, 1],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5, 316))

In [ ]:
binary_matrix

array([[1, 0, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 1, 1, 1],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5, 316))


## **Integer Sequence Matrices, Padding, and Truncation**

Bag-of-Words represents documents using word presence or frequency, but it does not preserve the order of words.

An alternative is to represent each document as an **ordered sequence of integer word IDs**.

For example:

$$
\mathbf{s}_1=[2,5,3]
$$

$$
\mathbf{s}_2=[4,2,1,5,3]
$$

These sequences preserve word order but have different lengths.

To organize them into a fixed-size matrix, we can apply:

- **Padding:** Add zeros to sequences shorter than the required length.
- **Truncation:** Remove tokens from sequences longer than the required length.

For a fixed sequence length $L=4$:

$$
S=
\begin{bmatrix}
2&5&3&0\\
4&2&1&5
\end{bmatrix}
$$

The resulting matrix has dimensions:

$$
S\in\mathbb{Z}_{\geq 0}^{D\times L}
$$

where:

- $D$ is the number of documents.
- $L$ is the fixed number of token positions.
- Each row represents an ordered token sequence.
- Zero represents padding rather than a vocabulary word.

We will apply this procedure to the integer-encoded paragraphs constructed in Part B.

**Applications:** Padded token-ID matrices are commonly used when preparing text for embedding layers and neural sequence models, including recurrent neural networks and Transformer-based architectures.

**Important:** Integer IDs are categorical identifiers. A padded token-ID matrix is not normally treated as a matrix of continuous numerical features. Neural models typically convert these IDs into learned vector representations using embedding layers.


In [ ]:

# Integer Sequence Matrix

# Use the same five paragraphs.
sequences = encoded_paragraphs[:5]

# Choose a fixed sequence length.
max_length = 30

# Initialize a matrix with zeros for padding.
sequence_matrix = np.zeros(
    (len(sequences), max_length),
    dtype=int
)

for i, sequence in enumerate(sequences):

    # Truncate long sequences.
    truncated = sequence[:max_length]

    # Copy token IDs; remaining positions stay zero.
    sequence_matrix[i, :len(truncated)] = truncated


In [ ]:
print("Number of documents:", len(sequences))
print("Fixed sequence length:", max_length)
print("Sequence matrix shape:", sequence_matrix.shape)

Number of documents: 5
Fixed sequence length: 30
Sequence matrix shape: (5, 30)


In [ ]:
print("\nFirst 3 rows and first 15 columns:")
print(sequence_matrix[:3, :15])


First 3 rows and first 15 columns:
[[268 230 278  21 181 290 140 295  39 264 159  67 245 236 299]
 [145 296   9 293 298 278 227  69 168  37 264 310 159   8 296]
 [136 272  31 296 182 167 202 245   8 292 278 257 145  12  91]]


In [ ]:
print("\nOriginal lengths:")
print([len(seq) for seq in sequences])

print("\nRetained token counts:")
print(np.count_nonzero(sequence_matrix, axis=1))

print("\nTruncated token counts:")
print([
    max(0, len(seq) - max_length)
    for seq in sequences
])


Original lengths:
[82, 134, 46, 0, 0]

Retained token counts:
[30 30 30  0  0]

Truncated token counts:
[52, 104, 16, 0, 0]



# **Exercise 2**: Preparing Real Text Data for Spam Detection

### Objective

In this exercise, you will apply the text preprocessing and numerical representation techniques learned in Parts B and C to a real-world spam detection problem.

You will prepare a dataset of labeled messages for a machine learning classifier.

The dataset contains two categories:

- **ham:** A legitimate message.
- **spam:** An unwanted or unsolicited message.

The machine learning training and prediction functions have already been provided. Your main task is to prepare the numerical input matrix correctly.

In [ ]:

#@title **Spam Detection Utilities (Run Once)** { display-mode: "form" }

import io
import zipfile
import urllib.request
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report


def load_spam_dataset():
    """Download the UCI SMS Spam Collection dataset."""

    url = (
        "https://archive.ics.uci.edu/static/public/228/"
        "sms+spam+collection.zip"
    )

    with urllib.request.urlopen(url, timeout=60) as response:
        archive_data = response.read()

    with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
        with archive.open("SMSSpamCollection") as file:
            df = pd.read_csv(
                file,
                sep="\t",
                header=None,
                names=["label", "message"],
                encoding="latin-1"
            )

    return df


def split_spam_dataset(df, test_size=0.2):
    """Split messages before vocabulary construction."""

    return train_test_split(
        df["message"].tolist(),
        df["label"].tolist(),
        test_size=test_size,
        random_state=42,
        stratify=df["label"]
    )


def train_spam_classifier(X_train, X_test, y_train, y_test):
    """Train and evaluate a Naive Bayes text classifier."""

    model = MultinomialNB()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print("Training matrix shape:", X_train.shape)
    print("Testing matrix shape:", X_test.shape)
    print(f"\nTest accuracy: {accuracy:.2%}")

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

    return model, accuracy


def predict_spam(model, messages, vectorizer):
    """Predict spam/ham for new raw text messages."""

    X_new = vectorizer.transform(messages)
    predictions = model.predict(X_new)

    results = pd.DataFrame({
        "message": messages,
        "prediction": predictions
    })

    return results


### Task 1: Load and explore the dataset

1. Download the dataset using `load_spam_dataset()`.
2. Display the first five messages.
3. Find the total number of messages.
4. Count the number of spam and ham messages.
5. Display two examples from each category.

**Hints:**
- Use `df.head()` to inspect the first rows.
- Use `len(df)` to count messages.
- Use `df["label"].value_counts()` to count categories.
- Use `df[df["label"] == "spam"]` to filter spam messages.

In [ ]:
# ============================================================
# Exercise 2: Spam Detection Using Text Representations
# ============================================================

import re
from sklearn.feature_extraction.text import CountVectorizer


# ------------------------------------------------------------
# Task 1: Load and explore the dataset
# ------------------------------------------------------------

df = load_spam_dataset()

print("First five messages:")
display(df.head())

# TODO: Print the total number of messages.
print("\nTotal messages:", len(df))

# TODO: Count spam and ham messages.
print("\nCategory counts:")
print(df["label"].value_counts())

# TODO: Display two spam messages.
print("\nSample spam messages:")
display(df[df["label"]=="spam"][:2])

# TODO: Display two ham messages.
print("\nSample ham messages:")
display(df[df["label"]=="ham"][:2])

First five messages:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Total messages: 5572

Category counts:
label
ham     4825
spam     747
Name: count, dtype: int64

Sample spam messages:


,label,message
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
5,spam,FreeMsg Hey there darling it's been 3 week's n...



Sample ham messages:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...


### Task 2: Normalize the messages

Create a function named `preprocess_message()` that:

1. Converts text to lowercase.
2. Replaces punctuation with spaces.
3. Replaces repeated whitespace with a single space.
4. Removes leading and trailing whitespace.

Apply your function to a few messages and compare the original and normalized versions.

**Hints:**
- Reuse the normalization techniques from Part B.
- `text.lower()` converts text to lowercase.
- `re.sub(r"[^\w\s]", " ", text)` replaces punctuation.
- `re.sub(r"\s+", " ", text)` normalizes whitespace.
- `text.strip()` removes surrounding whitespace.

**Question:** Why is it important to apply the same preprocessing to both training and testing messages?

In [ ]:
# ------------------------------------------------------------
# Task 2: Normalize the text
# ------------------------------------------------------------

def preprocess_message(text):
    """Normalize a text message."""

    # TODO: Convert to lowercase.
    text = text.lower()

    # TODO: Replace punctuation with spaces.
    text = re.sub(r"[^\w\s]"," ",text)

    # TODO: Replace repeated whitespace with one space.
    text = re.sub(r"[\s+]"," ", text)

    # TODO: Remove leading and trailing whitespace.

    return text.strip()


print("\nOriginal message:")
print(df["message"].iloc[0])

print("\nNormalized message:")

print(preprocess_message(df["message"].iloc[0]))



Original message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Normalized message:
go until jurong point  crazy   available only in bugis n great world la e buffet    cine there got amore wat


### Task 3: Split the dataset

Use the provided `split_spam_dataset()` function to create training and testing sets.

The training messages will be used to construct the vocabulary. Testing messages must not be used to learn the vocabulary.

**Question:** Why should we construct the vocabulary using only the training data?


In [ ]:
# ------------------------------------------------------------
# Task 3: Split training and testing data
# ------------------------------------------------------------

X_train_text, X_test_text, y_train, y_test = (
    split_spam_dataset(df)
)

print("\nTraining messages:", len(X_train_text))
print("Testing messages:", len(X_test_text))



Training messages: 4457
Testing messages: 1115


### Task 4: Construct the document-term matrices

Use `CountVectorizer` from scikit-learn to convert the messages into count-based Bag-of-Words vectors.

1. Create a vectorizer using your preprocessing function.
2. Fit the vectorizer on the training messages.
3. Transform the testing messages using the same vectorizer.
4. Inspect the shapes of the resulting matrices.
5. Display the first 15 vocabulary words.
6. Inspect the nonzero word counts of one training message.

**Hints:**
- `CountVectorizer` automates the count-based document-term matrix construction you performed manually in Part C.
- `fit_transform()` learns the vocabulary and constructs the training matrix.
- `transform()` constructs a matrix using the existing vocabulary.
- `get_feature_names_out()` returns the vocabulary in matrix-column order.
- `.toarray()` can convert a small slice of a sparse matrix into a dense NumPy array.

**Important:** Do not call `fit_transform()` on the testing messages.


In [ ]:
# ------------------------------------------------------------
# Task 4: Construct document-term matrices
# ------------------------------------------------------------

# TODO: Create a CountVectorizer using preprocess_message.
# Hint: Use the preprocessor= argument.
vectorizer = CountVectorizer(preprocessor=preprocess_message)

# TODO: Learn vocabulary from training messages and transform them.
X_train = vectorizer.fit_transform(X_train_text)

# TODO: Transform testing messages using the same vocabulary.
X_test = vectorizer.transform(X_test_text)

print("\nTraining matrix shape:", X_train.shape)
print("Testing matrix shape:", X_test.shape)

# TODO: Obtain vocabulary words in column order.
feature_names = vectorizer.get_feature_names_out()

print("\nFirst 15 vocabulary words:")
print(feature_names[:15])

# Inspect a small portion of the first document vector.
print("\nFirst training document, first 20 columns:")
print(X_train[0, :20].toarray())

# TODO: Calculate the number of nonzero entries in X_train.
print("\nNumber of nonzero matrix entries:", )


Training matrix shape: (4457, 7699)
Testing matrix shape: (1115, 7699)

First 15 vocabulary words:
['00' '000' '000pes' '008704050406' '0089' '0121' '01223585236'
 '01223585334' '02' '0207' '02072069400' '02073162414' '02085076972' '021'
 '03']

First training document, first 20 columns:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]

Number of nonzero matrix entries:


### Task 5: Train and evaluate the classifier

Pass your prepared matrices and labels to the provided function:

`train_spam_classifier(X_train, X_test, y_train, y_test)`

The function will train a Naive Bayes classifier and report its accuracy.

You do not need to implement the machine learning algorithm.


In [ ]:
# ------------------------------------------------------------
# Task 5: Train and evaluate the classifier
# ------------------------------------------------------------

# Instructor-provided function: no ML implementation required.
model, accuracy = train_spam_classifier(
    X_train,
    X_test,
    y_train,
    y_test
)


Training matrix shape: (4457, 7699)
Testing matrix shape: (1115, 7699)

Test accuracy: 98.74%

Classification report:
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



### Task 6: Predict new messages

Use the provided `predict_spam()` function to classify new messages.

The function will preprocess and vectorize the messages using the fitted vectorizer before making predictions.


In [ ]:
# ------------------------------------------------------------
# Task 6: Predict new messages
# ------------------------------------------------------------

new_messages = [
    "Congratulations! You have won a free cash prize. Claim now!",
    "Hi, are we still meeting tomorrow afternoon?",
    "URGENT! Call this number to receive your reward.",
    "Please send me the notes from today's lecture."
]

# Instructor-provided prediction function.
prediction_results = predict_spam(
    model,
    new_messages,
    vectorizer
)

print("\nPredictions for new messages:")
display(prediction_results)


Predictions for new messages:


,message,prediction
0,Congratulations! You have won a free cash priz...,spam
1,"Hi, are we still meeting tomorrow afternoon?",ham
2,URGENT! Call this number to receive your reward.,spam
3,Please send me the notes from today's lecture.,ham


### Questions

1. What does each row of `X_train` represent?
2. What does each column represent?
3. Why is `X_train` usually a sparse matrix?
4. What would happen if the vocabulary columns were arranged differently during testing?
5. Why must a new message be transformed before it can be passed to the classifier?
6. Does count-based Bag-of-Words preserve the original word order?

In [ ]:
# ------------------------------------------------------------
# Written questions
# ------------------------------------------------------------

#Q1: What does each row of X_train represent?
print(f"Q1:row represents a test data that is going to labeled in our case a message to be classified as spam or not spam")

# Q2: What does each column represent?
# Answer:
print(f"Q2:column represent catagory of our row data.")
# Q3: Why is X_train usually sparse?
# Answer:
print(f"Q3:cause a single feature  uses li ttle fraction of the total vocabulary ")

# Q4: Why must training and testing use the same vocabulary?
# Answer:
print(f"Q4:For consistency of our model performance in test, train and real word prediction ")
# Q5: Why must new messages be transformed before prediction?
# Answer:
print(f"Q5:Because our model works on transformed data not on row so we need to transform")

# Q6: Does count-based Bag-of-Words preserve word order?
# Answer:
print(f"Q6:No only count of words or frequency not order")

Q1:row represents a test data that is going to labeled in our case a message to be classified as spam or not spam
Q2:column represent catagory of our row data.
Q3:cause a single feature  uses li ttle fraction of the total vocabulary 
Q4:For consistency of our model performance in test, train and real word prediction 
Q5:Because our model works on transformed data not on row so we need to transform
Q6:No only count of words or frequency not order
